# 🚀 50M Bengali GPT - Production Scratch Pretraining
### ⚡ হাইলাইট ও স্পেসিফিকেশন:
- **প্যারামিটার:** ~54.3 Million (১০০% আনফ্রোজেন, প্রতিটি নিউরন সক্রিয়ভাবে শিখবে)
- **কনটেক্সট লেন্থ:** 512 Tokens (~৩০০-৩৫০টি বাংলা শব্দ, ৩ গুণ দ্রুত ট্রেনিং!)
- **ভোকাবুলারি:** 10,000 (ByteLevel BPE, ~1.85 tokens/word)
- **ডেটাসেট:** বাংলা উইকিপিডিয়া (Wikimedia/Wikipedia বাংলা ডাম্প) থেকে সরাসরি ডাউনলোড ও প্রসেসকৃত ২,৫০,০০০+ লাইন বিশুদ্ধ বাংলা কর্পাস!
- **ট্রেনিং ও ভ্যালিডেশন:** ৯০% Train ও ১০% Validation ট্র্যাকিং (মুখস্থ রোধ ও নিখুঁত শিখন যাচাই)
- **হার্ডওয়্যার:** Colab Free T4 GPU (~1.0 GB VRAM খরচ, 14 GB মেমরি নিরাপদ)

In [ ]:
# Step 1: GPU চেক করুন (NVIDIA T4 থাকা নিশ্চিত করুন)
!nvidia-smi

In [ ]:
# Step 2: Google Drive মাউন্ট করুন (চেকপয়েন্ট ব্যাকআপের জন্য)
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/bengali_gpt_50m_checkpoints'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f"✓ Drive Checkpoint ডিরেক্টরি প্রস্তুত: {DRIVE_CHECKPOINT_DIR}")

In [ ]:
# Step 3: রিপোজিটরি ক্লোন ও প্রয়োজনীয় লাইব্রেরি ইনস্টল করুন
import os

%cd /content
!rm -rf ss_100m
!git clone https://github.com/kajshikhi49-afk/ss_100m.git

%cd /content/ss_100m/ss_50million
!pip install -q tokenizers torch numpy datasets pyarrow
print("✓ এনভায়রনমেন্ট ও লাইব্রেরি প্রস্তুত!")

In [ ]:
# Step 4: 📥 বাংলা উইকিপিডিয়া থেকে ২,৫০,০০০ লাইন ডেটা ডাউনলোড ও প্রসেসিং
# উইকিমিডিয়া অফিশিয়াল বাংলা ডাম্প থেকে পরিষ্কার টেক্সট সংগ্রহ
import os
import re
import random
from datasets import load_dataset

corpus_file = 'data/corpus.txt'

if os.path.exists(corpus_file) and os.path.getsize(corpus_file) > 10 * 1024 * 1024:
    print(f"✓ বিদ্যমান বিশাল কর্পাস পাওয়া গেছে ({os.path.getsize(corpus_file)/(1024*1024):.1f} MB)")
else:
    print("⚡ বাংলা উইকিপিডিয়া থেকে ডেটাসেট ডাউনলোড ও প্রসেস করা হচ্ছে...")
    wiki_ds = load_dataset('wikimedia/wikipedia', '20231101.bn', split='train', streaming=True)
    
    lines = []
    TARGET_LINES = 250000  # ২.৫ লাখ লাইন বিশুদ্ধ বাংলা ডেটা
    
    for item in wiki_ds:
        text = item.get('text', '')
        for paragraph in text.split('\n'):
            p = paragraph.strip()
            # ফিল্টার: শুধু অর্থপূর্ণ ও পরিষ্কার বাংলা লাইন নেওয়া
            if len(p) >= 30 and re.search(r'[\u0980-\u09FF]', p):
                lines.append(p)
                if len(lines) >= TARGET_LINES:
                    break
        if len(lines) >= TARGET_LINES:
            break
    
    print(f"✓ সংগৃহীত হয়েছে: {len(lines):,} লাইন বাংলা টেক্সট!")
    
    # শাফল করা যাতে মডেল মুখস্থ না করে বিভিন্ন বিষয়ের বাক্য শেখে
    print("⚡ ডেটা এলোমেলো (Shuffle) করা হচ্ছে...")
    random.seed(42)
    random.shuffle(lines)
    
    os.makedirs('data', exist_ok=True)
    with open(corpus_file, 'w', encoding='utf-8') as f:
        for l in lines:
            f.write(l + '\n')
    
    # পুরোনো বাইনারি ক্যাশ মুছে ফেলা যাতে নতুন ডেটা থেকে ফ্রেশ ক্যাশ হয়
    if os.path.exists('data/corpus_tokens.bin'):
        os.remove('data/corpus_tokens.bin')
    
    print(f"✓ সফলভাবে সংরক্ষিত: {corpus_file} ({os.path.getsize(corpus_file)/(1024*1024):.1f} MB)")

In [ ]:
# Step 5: ⚡ ২.৫ লাখ ডেটা থেকে নতুন 10,000 Vocab BPE টোকেনাইজার পুনঃপ্রশিক্ষণ
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

special_tokens = ['<PAD>', '<UNK>', '<BOS>', '<EOS>', '<|system|>', '<|user|>', '<|assistant|>', '<|math|>']
tok = Tokenizer(models.BPE(unk_token='<UNK>'))
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False)
tok.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=10000,
    special_tokens=special_tokens,
    min_frequency=2,
    show_progress=True
)

print("⚡ নতুন ২,৫০,০০০ লাইন ডেটার ওপর টোকেনাইজার ট্রেনিং শুরু হচ্ছে...")
tok.train(['data/corpus.txt'], trainer)
tok.save('tokenizer.json')
print(f"✓ টোকেনাইজার তৈরি ও সেভ সম্পন্ন! Vocab Size: {tok.get_vocab_size():,}")

# টেস্ট ডিকোড
sample = "বাংলা ভাষা হলো দক্ষিণ এশিয়ার অন্যতম সমৃদ্ধ ও সুন্দর ভাষা।"
enc = tok.encode(sample)
words = sample.split()
print(f"টেক্সট: {sample}")
print(f"শব্দ: {len(words)} টি | টোকেন: {len(enc.ids)} টি | অনুপাত: {len(enc.ids)/len(words):.2f} tokens/word")

In [ ]:
# Step 6: মডেল কনফিগারেশন যাচাই করুন (Context 512 + 10k Vocab)
from src.config import GPTConfig
from src.model import BengaliGPT as GPT

p = GPTConfig.estimate_parameters()
print(f"মডেল মোট প্যারামিটার: {p['total_untied']/1e6:.2f}M")
print(f"কনটেক্সট লেন্থ (Block Size): {GPTConfig.block_size} টোকেন (৩ গুণ দ্রুত গতি!)")
print(f"মাইক্রো-ব্যাচ: {GPTConfig.batch_size}, একিউমুলেশন: {GPTConfig.gradient_accumulation_steps}")
print(f"ইফেক্টিভ ব্যাচ সাইজ: {GPTConfig.batch_size * GPTConfig.gradient_accumulation_steps}")

In [ ]:
# Step 7: 🚀 প্রোডাকশন স্ক্র্যাচ প্রি-ট্রেনিং (Train ও Val Loss ট্র্যাকিং সহ)
import os
import sys
import time
import math
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from src.config import GPTConfig
from src.model import BengaliGPT as GPT
from src.dataset import BengaliDataset
from tokenizers import Tokenizer

# ১. ডিভাইস ও GPU অপটিমাইজেশন
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ২. টোকেনাইজার ও ডেটাসেট (৯০% Train, ১০% Val)
tokenizer = Tokenizer.from_file("tokenizer.json")
corpus_file = "data/corpus.txt"
dataset = BengaliDataset(corpus_path=corpus_file, tokenizer=tokenizer, block_size=GPTConfig.block_size, split_ratio=0.9)
print("✓ ডেটাসেট মেমরি-ম্যাপ প্রস্তুত!")

# ৩. মডেল ইনিশিয়ালাইজেশন (১০০% আনফ্রোজেন, ফুল লার্নিং)
raw_model = GPT(GPTConfig).to(device)
total_params = sum(p.numel() for p in raw_model.parameters())
trainable_params = sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
print(f"✓ মডেল প্রস্তুত! মোট প্যারামিটার: {total_params/1e6:.2f}M (১০০% আনফ্রোজেন)")

# PyTorch 2.0 কম্পাইল অপটিমাইজেশন
try:
    model = torch.compile(raw_model)
    print("✓ torch.compile সক্রিয়!")
except Exception as e:
    model = raw_model

# ৪. অপটিমাইজার ও শিডিউলার
optimizer = torch.optim.AdamW(raw_model.parameters(), lr=GPTConfig.learning_rate, betas=(0.9, 0.95), weight_decay=0.1)
scaler = GradScaler()

def get_lr(it, max_iters=5000, warmup_iters=250, lr=3e-4, min_lr=3e-5):
    if it < warmup_iters:
        return lr * it / warmup_iters
    if it > max_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (lr - min_lr)

@torch.no_grad()
def estimate_val_loss(eval_iters=20):
    raw_model.eval()
    losses = []
    for _ in range(eval_iters):
        x, y = dataset.get_batch('val', batch_size=GPTConfig.batch_size, device=device)
        with autocast(dtype=torch.float16):
            _, loss = raw_model(x, y)
        losses.append(loss.item())
    raw_model.train()
    return sum(losses) / len(losses)

# ৫. ট্রেনিং লুপ
print("=" * 65)
print("🔥 ফুল স্ক্র্যাচ ট্রেনিং শুরু হচ্ছে (প্রতি ৫০০ স্টেপে ড্রাইভ ব্যাকআপ)... ")
print("=" * 65)

start_time = time.time()
model.train()
optimizer.zero_grad(set_to_none=True)

for step in range(1, GPTConfig.max_iters + 1):
    current_lr = get_lr(step, max_iters=GPTConfig.max_iters)
    for param_group in optimizer.param_groups:
        param_group['lr'] = current_lr

    accum_loss = 0.0
    for micro_step in range(GPTConfig.gradient_accumulation_steps):
        x, y = dataset.get_batch('train', batch_size=GPTConfig.batch_size, device=device)
        with autocast(dtype=torch.float16):
            logits, loss = model(x, y)
            loss = loss / GPTConfig.gradient_accumulation_steps
        scaler.scale(loss).backward()
        accum_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    # লগ এবং ভ্যালিডেশন লস হিসাব (প্রতি ২৫০ স্টেপে)
    if step % 250 == 0 or step == 1:
        elapsed = time.time() - start_time
        speed = step / elapsed if elapsed > 0 else 0
        val_loss = estimate_val_loss(eval_iters=15)
        print(f"Step {step:4d}/{GPTConfig.max_iters} | Train Loss: {accum_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {current_lr:.2e} | Speed: {speed:.2f} it/s")

    # প্রতি ৫০০ স্টেপ পরপর Google Drive-এ সেভ
    if step % GPTConfig.save_interval == 0 or step == GPTConfig.max_iters:
        ckpt_path = os.path.join(DRIVE_CHECKPOINT_DIR, f"bengali_gpt_50m_step_{step}.pt")
        torch.save({
            'step': step,
            'model_state_dict': raw_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': accum_loss,
            'config': GPTConfig
        }, ckpt_path)
        print(f"💾 [SAVED] চেকপয়েন্ট ড্রাইভে সেভ হয়েছে: {ckpt_path}")

print("🎉 ৫০M মডেলের ফুল প্রি-ট্রেনিং সফলভাবে সম্পন্ন হয়েছে!")

In [ ]:
# Step 8: 💬 আপনার ইচ্ছামতো প্রম্পট লিখে সরাসরি মডেল পরীক্ষা করুন!
eval_model = raw_model if 'raw_model' in locals() else model
eval_model.eval()

my_custom_prompt = "বাংলা ভাষা ও সাহিত্য হলো"  # <-- এখানে আপনার যেকোনো প্রম্পট লিখুন

enc = tokenizer.encode(my_custom_prompt)
ids = enc.ids if hasattr(enc, 'ids') else enc
input_tensor = torch.tensor([ids], dtype=torch.long, device=device)

with torch.no_grad():
    out = eval_model.generate(
        input_tensor,
        max_new_tokens=150,        # উত্তরের দৈর্ঘ্য
        temperature=0.75,          # ক্রিয়েটিভিটি (0.7-0.8 আদর্শ)
        top_k=40,                  # অপ্রাসঙ্গিক শব্দ ফিল্টার
        repetition_penalty=1.25    # পুনরাবৃত্তি আটকানো
    )

reply = tokenizer.decode(out[0].cpu().tolist())
print("=" * 60)
print("🤖 মডেলের উত্তর:")
print(reply)
print("=" * 60)